# Customer Intelligence & Support Copilot
## Instructor-led end-to-end notebook

**Scenario:** synthetic grocery quick-commerce data. This notebook emphasizes *how data is fetched, inspected, cleaned, validated, stored, and then consumed by downstream AI/ML components*.

### Before class
1. Run `python generate_data.py` from the repository root.
2. Install packages with `pip install -r requirements.txt`.
3. Run this notebook from the repository root so relative paths work.

All data and policies are fictional. The LLM calls are handled by `mock_llm.py` locally; no API key is required.

## 0. Repository map and end-to-end flow

`generate_data.py` → `data/raw/*.csv` → audit → clean → `data/clean/*.csv` + `data/instamart.db` → RFM/segments + lapse model → privacy-safe ticket triage → policy retrieval + refund tools.

Teaching point: **fetching data** does not always mean calling an API. Here, the source is a local CSV export. In a production system, the same ingestion layer could use an approved API or warehouse connector, but credentials and network access are intentionally out of scope.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "generate_data.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)
print("Root files:", sorted(p.name for p in ROOT.iterdir() if p.is_file()))

## 1. Generate / fetch the raw data

The data generator is deterministic (fixed random seed). It writes customer, product, order, support-ticket, evaluation-question, agent-scenario inputs and policy text files.

Run it once when starting from a clean clone. Running it again recreates the same synthetic source exports.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, str(ROOT / "generate_data.py")], check=True)
raw_dir = ROOT / "data" / "raw"
print("Raw files:", sorted(p.name for p in raw_dir.iterdir()))

## 2. Read the exports and understand their shape

Use `keep_default_na=False` for ticket text so empty strings remain empty strings. This is important because empty tickets must be identified by an explicit guardrail rather than silently changing the input representation.

In [ ]:
import pandas as pd
customers_raw = pd.read_csv(raw_dir / "customers.csv", keep_default_na=False)
products_raw = pd.read_csv(raw_dir / "products.csv", keep_default_na=False)
orders_raw = pd.read_csv(raw_dir / "orders.csv", keep_default_na=False)
tickets_raw = pd.read_csv(raw_dir / "tickets.csv", keep_default_na=False)

for name, df in [("customers", customers_raw), ("products", products_raw),
                 ("orders", orders_raw), ("tickets", tickets_raw)]:
    print(f"\\n{name}: shape={df.shape}")
    display(df.head(3))
    display(pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_count": df.isna().sum(),
        "blank_count": df.astype(str).apply(lambda col: col.str.strip().eq("").sum()),
        "missing_pct": (df.isna().mean() * 100).round(2)
    }))

### Audit questions for students
- Which columns are supposed to be unique keys?
- Are duplicates exact duplicates or multiple states of the same order?
- Which values are impossible (negative delivery duration, non-numeric money, implausible age)?
- Why should we distinguish an empty string from a missing value?
- What would go wrong if we modelled before cleaning?

In [ ]:
from src.data_pipeline import audit, clean_data, validate
audit()

## 3. Clean the data and write trusted outputs

The pipeline applies explicit rules:
- standardize city and gender;
- make impossible ages missing and fill with city median;
- parse timestamps, remove exact duplicates, then keep the latest row per order ID;
- coerce invalid amounts to missing and calculate replacements from product price × quantity × discount;
- set negative delivery times to missing and impute only completed orders using product-category medians;
- save clean CSVs and SQLite tables.

Ask students to explain why cancelled orders should keep a missing delivery duration.

In [ ]:
clean = clean_data()
for name, df in clean.items():
    print(name, df.shape)
    display(df.head(3))
validate()

## 4. Query the SQLite database

The clean tables are persisted in `data/instamart.db`. SQLite gives us a portable analytical database that can be queried without setting up a server.

Run the supplied SQL examples and compare SQL totals to pandas results.

In [ ]:
import sqlite3
db_path = ROOT / "data" / "instamart.db"
with sqlite3.connect(db_path) as con:
    tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", con)
    display(tables)
    q2 = pd.read_sql_query(\"\"\"
        SELECT c.city, p.category, ROUND(SUM(o.order_amount), 2) AS revenue
        FROM orders o
        JOIN customers c USING(customer_id)
        JOIN products p USING(product_id)
        WHERE o.status = 'completed'
        GROUP BY c.city, p.category
        ORDER BY c.city, p.category
    \"\"\", con)
display(q2)
print("SQL grand total INR:", round(q2.revenue.sum(), 2))

## 5. Part B — customer segments and lapse prediction

RFM means:
- **Recency:** days since the most recent completed order at the snapshot date.
- **Frequency:** number of distinct completed orders.
- **Monetary:** total spend across completed orders.

The model module scales RFM for K-means, compares K values, uses PCA to inspect compressed variance, and builds a lapse model. The lapse model's input features use only orders before 2025-09-01; September is reserved to define the label. The scaler and encoder are fitted inside a training pipeline, not on the test data.

In [ ]:
from src.customer_models import rfm_and_segments, build_lapse_model
rfm, k_diagnostics, segment_profile, pca_variance = rfm_and_segments(k=4)
display(k_diagnostics.round(3))
display(segment_profile.round(2))
print("PCA explained variance ratio:", pca_variance, "total:", pca_variance.sum())
display(rfm.head())
lapse_summary = build_lapse_model()
lapse_summary

### Instructor discussion: leakage
Ask students to identify:
1. Which dates can enter the lapse-model features?
2. Why must September orders not enter the features?
3. Why is a Premium-only retention list labelled *illustrative* if the Premium flag comes from whole-window RFM?
4. Why can accuracy be misleading when a substantial minority of customers lapse?

## 6. Part C — support ticket triage

This pipeline demonstrates:
1. input guardrails for empty and overlong tickets;
2. ordered PII redaction (email → phone → order ID);
3. retry logic for simulated transient errors;
4. strict category validation and a keyword fallback;
5. token budgeting and human-review routing.

The exact user message sent to the mock classifier is the redacted ticket text only.

In [ ]:
from src.triage import run_pipeline, FEW, ZERO
triage_results, triage_summary = run_pipeline(prompt=FEW, token_budget=100000)
print(triage_summary)
display(triage_results.head(10))
display(triage_results.status.value_counts().rename_axis("status").reset_index(name="tickets"))

### Discussion: privacy and safety
- Redaction is a defence-in-depth step, not a guarantee that every sensitive detail is removed.
- Why are invalid categories sent to human review instead of guessed?
- Why should fraud override ordinary customer-segment routing?
- Why do empty/long tickets need to be blocked before any model call?

## 7. Part D — policy RAG

The retrieval layer chunks policy documents, fits a TF-IDF vectorizer and ranks chunks by cosine similarity. It passes retrieved chunks to the offline policy assistant, which should answer only from the context and fall back when a question is not covered.

In [ ]:
from src.policy_rag import PolicyRAG, evaluate
rag = PolicyRAG(chunk_size=40, overlap=10)
print("Policy chunks:", len(rag.chunks))
display(pd.DataFrame(rag.chunks).head())
print(rag.answer("What delivery time makes a completed order eligible for a full refund?"))
policy_eval = evaluate(rag)
display(policy_eval)
print("Expected-document hits:", int(policy_eval.retrieved_doc.sum()))
print("Expected-fact hits:", int(policy_eval.fact_found.sum()))
print("Fallbacks:", int(policy_eval.fallback.sum()))

## 8. Part D — refund tools and safe stopping

The refund agent demonstrates a tool loop: look up an order in SQLite, retrieve policy, calculate the eligible refund, then issue it or escalate to a human. In this teaching kit the ledger is a local JSONL file; no money moves.

Before running agent demonstrations, inspect `src/refund_agent.py` and ask students to trace how a tool name maps to a Python function.

In [ ]:
from src.refund_agent import get_order, search_policy, calculate_refund, TOOL_MAP, run_agent
print("Tools:", sorted(TOOL_MAP))
print("Example order lookup:")
with sqlite3.connect(db_path) as con:
    example_order = pd.read_sql_query(
        "SELECT order_id FROM orders WHERE status='completed' AND delivery_minutes > 45 LIMIT 1", con
    ).iloc[0, 0]
print(example_order, get_order(example_order))
print("Refund calculation:", calculate_refund(250.0, 52))
print("Timeout safety demo:", run_agent(f"Refund order {example_order}", scenario_id="timeout-demo", time_limit_seconds=0))

## 9. Final generated artifacts and reproducibility check

Review the CSVs and model file generated by the pipeline. In class, have students delete `data/clean/`, `data/instamart.db`, and `outputs/`, then rerun the documented commands to verify that the project can be rebuilt from source.

**Important:** This teaching kit is an instructional implementation. If the course grades exact numeric checkpoints from the original brief, run every checkpoint and reconcile differences; do not hardcode expected answers.

In [ ]:
for folder in [ROOT/"data"/"clean", ROOT/"outputs", ROOT/"state"]:
    print(f"\\n{folder.relative_to(ROOT)}")
    print(sorted(p.name for p in folder.iterdir()) if folder.exists() else "(not created)")
print("Database exists:", db_path.exists())